In [ ]:
# Install hypertools (dev-1.0-refactor preview) -- run this first on Colab.
# On release this becomes: %pip install hypertools
%pip install -q "hypertools[interactive] @ git+https://github.com/ContextLab/hypertools.git@dev-1.0-refactor"

%matplotlib inline


# Story trajectories: brain activity while listening to a story

This example walks through the "story trajectories" demo (GH #275): an
animated, hyperaligned point cloud that shows how each subject's
whole-brain activity pattern moves through a shared, low-dimensional space
over the course of a spoken story.

## Background

The data (``hyp.load('weights')``) come from Simony et al. (2016,
*Nature Communications*, [10.1038/ncomms12141](https://doi.org/10.1038/ncomms12141)), an fMRI study in which subjects
listened to the same ~7-minute spoken story -- "PieMan", told live by Jim
O'Grady at a Moth GrandSLAM event -- while their whole-brain activity was
recorded. Each subject's raw voxel-by-timepoint data were summarized with
Hierarchical Topographic Factor Analysis (HTFA) into ``k=100`` latent
"hubs" -- spatially compact, story-timescale sources of correlated
activity -- giving every subject a ``(timepoints, 100)`` trajectory through
"hub space" that HyperTools can align, reduce, and animate directly.

## What the animation shows

Every subject starts the story with an idiosyncratic, unaligned activity
pattern. Hyperalignment (``align='HyperAlign'``) rotates every subject's
trajectory into a common space that maximizes shared, story-locked
structure -- so that, once aligned, subjects' points move *together*
through the space as the story unfolds, tracing out a shared path shaped by
the story's narrative structure (not a straight line -- see the
"interesting, non-linear paths" check in the evidence script). This is the
real acceptance criterion from GH #275: after alignment, subjects' point
clouds should move noticeably more in sync than before alignment, without
literally replicating any specific published figure.

## The exact code

The full pipeline -- manip (smooth + resample + z-score) -> hyperalign ->
UMAP -> animate -- takes a few minutes to run on the full dataset (mostly
UMAP), so this example does **not** re-run it live; instead it displays the
pre-rendered result from ``docs/images/v1.0-round17/`` (regenerated by
``scripts/round17_evidence/story_trajectories.py``). Here is the exact code
that produced it:

```python
import hypertools as hyp

data = hyp.load('weights')

# smooth each subject's timeseries, resample everyone onto a common
# 1000-sample grid, then z-score -- per-dataset preprocessing in
# native (100-dim hub) space, BEFORE alignment/reduction (GH #153)
manip_spec = [
    {'model': 'Smooth', 'kwargs': {'kernel_width': 25}},
    {'model': 'Resample', 'kwargs': {'n_samples': 1000}},
    'ZScore',
]

hyp.plot(
    data,
    manip=manip_spec,
    align={'model': 'HyperAlign', 'kwargs': {'n_iter': 10}},
    reduce='UMAP',
    animate='window', duration=30, focused=4,
    save_path='story_trajectories.mp4',
)
```
``animate='window'`` plays a moving "focused" trail of the last ``focused``
timepoints per subject (rather than the whole path at once) while slowly
spinning the camera, so the story's temporal structure is visible frame by
frame.

Below: three representative frames -- early, middle, and late in the story
-- followed by the full animation
(``docs/images/v1.0-round17/story_trajectories.mp4``).

.. video:: /images/v1.0-round17/story_trajectories.mp4
   :width: 700
   :loop:


In [ ]:
# Code source: Contextual Dynamics Laboratory
# License: MIT

import os

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

import hypertools as hyp


def _find_img_dir():
    """Locate docs/images/v1.0-round17 regardless of how this script is
    run: directly (`__file__` is defined), or execed by sphinx-gallery
    (which chdir's into this script's own directory before running it, but
    does NOT define `__file__` -- so `os.getcwd()` stands in for it)."""
    bases = []
    try:
        bases.append(os.path.dirname(os.path.abspath(__file__)))
    except NameError:
        pass
    bases.append(os.getcwd())
    bases.append(os.path.dirname(os.path.abspath(hyp.__file__)))
    for base in bases:
        for up in ('..', os.path.join('..', '..')):
            candidate = os.path.normpath(
                os.path.join(base, up, 'docs', 'images', 'v1.0-round17'))
            if os.path.isdir(candidate):
                return candidate
    raise RuntimeError('could not locate docs/images/v1.0-round17')


IMG_DIR = _find_img_dir()

# three representative frames -- early, middle, and late in the story --
# from the pre-rendered animation
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, label in zip(axes, ('early', 'mid', 'late')):
    img = mpimg.imread(os.path.join(IMG_DIR, f'story_frame_{label}.png'))
    ax.imshow(img)
    ax.set_title(f'{label} in the story')
    ax.axis('off')
plt.tight_layout()
plt.show()